# SiPM Centroid Baseline

This notebook evaluates simple count-weighted centroid localization using the 32 SiPM counts. It compares two methods:

- `all_32_2d_centroid`: treat all 32 SiPMs as physical 2D points and compute a weighted centroid.
- `projection_centroid`: use `sipm_100..115` as the x projection and `sipm_300..315` as the z projection.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "macros").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAINING_DIR = PROJECT_ROOT / "analysis/training_scan_data_1024"
TEST_DIR = PROJECT_ROOT / "analysis/test_scan_data_1000"
MACRO_PATH = PROJECT_ROOT / "macros/muon_scan.mac"

SUMMARY_CSV = PROJECT_ROOT / "analysis/centroid_sipm_summary.csv"
PREDICTIONS_CSV = PROJECT_ROOT / "analysis/centroid_sipm_predictions.csv"
FIGURE_DIR = PROJECT_ROOT / "analysis/centroid_figures"
FIGURE_DIR.mkdir(exist_ok=True)

RUN_FILE_RE = re.compile(r"MUON-run(?P<run>\d+)_sipm_counts_by_event\.csv$")
X_ROW_COPIES = tuple(range(100, 116))
Z_ROW_COPIES = tuple(range(300, 316))
SIPM_COPIES = X_ROW_COPIES + Z_ROW_COPIES
SIPM_COLUMNS = [f"sipm_{copy}" for copy in SIPM_COPIES]

LANE_START_CM = -46.875
LANE_PITCH_CM = 6.25
SIPM_EDGE_CM = 51.035
EPS = 1e-12

## Load Data

In [ ]:
def parse_muon_scan_positions(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5 and parts[0] == "/gps/position":
            x_cm, _, z_cm = map(float, parts[1:4])
            rows.append({"run_id": len(rows), "true_x_cm": x_cm, "true_z_cm": z_cm})
    return pd.DataFrame(rows)


def load_training_events(path: Path, labels: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(path.glob("MUON-run*_sipm_counts_by_event.csv")):
        match = RUN_FILE_RE.match(csv_path.name)
        if not match:
            continue
        df = pd.read_csv(csv_path)
        df = df[["EventID", *SIPM_COLUMNS]].copy()
        df.insert(0, "run_id", int(match.group("run")))
        df = df.rename(columns={"EventID": "event_id"})
        frames.append(df)
    events = pd.concat(frames, ignore_index=True)
    return events.merge(labels, on="run_id", validate="many_to_one")


def load_test_events(path: Path) -> pd.DataFrame:
    manifest = pd.read_csv(path / "test_manifest.csv")
    rows = []
    for _, meta in manifest.sort_values("event_id").iterrows():
        df = pd.read_csv(path / meta["output_file"])
        row = df.iloc[0].to_dict()
        row["event_id"] = int(meta["event_id"])
        row["run_id"] = int(meta["run_id"])
        row["true_x_cm"] = float(meta["true_x_cm"])
        row["true_z_cm"] = float(meta["true_z_cm"])
        rows.append(row)
    return pd.DataFrame(rows).sort_values("event_id").reset_index(drop=True)


labels = parse_muon_scan_positions(MACRO_PATH)
training = load_training_events(TRAINING_DIR, labels)
test = load_test_events(TEST_DIR)
print(len(training), len(test))

## Centroid Calculation

In [ ]:
def lane_coordinate(copy_number: int) -> float:
    if 100 <= copy_number <= 115:
        return LANE_START_CM + (copy_number - 100) * LANE_PITCH_CM
    if 300 <= copy_number <= 315:
        return LANE_START_CM + (copy_number - 300) * LANE_PITCH_CM
    raise ValueError(copy_number)


def sipm_positions() -> pd.DataFrame:
    rows = []
    for copy in X_ROW_COPIES:
        rows.append({"copy": copy, "x_cm": lane_coordinate(copy), "z_cm": SIPM_EDGE_CM, "family": "+z row"})
    for copy in Z_ROW_COPIES:
        rows.append({"copy": copy, "x_cm": SIPM_EDGE_CM, "z_cm": lane_coordinate(copy), "family": "+x row"})
    return pd.DataFrame(rows)


def add_centroid_predictions(events: pd.DataFrame, dataset: str) -> pd.DataFrame:
    out = events.copy()
    counts = out[SIPM_COLUMNS].to_numpy(dtype=np.float64)
    positions = sipm_positions()
    total = np.maximum(counts.sum(axis=1), EPS)
    out["pred_x_all_centroid_cm"] = counts.dot(positions["x_cm"].to_numpy()) / total
    out["pred_z_all_centroid_cm"] = counts.dot(positions["z_cm"].to_numpy()) / total

    x_counts = out[[f"sipm_{copy}" for copy in X_ROW_COPIES]].to_numpy(dtype=np.float64)
    z_counts = out[[f"sipm_{copy}" for copy in Z_ROW_COPIES]].to_numpy(dtype=np.float64)
    x_lanes = np.array([lane_coordinate(copy) for copy in X_ROW_COPIES])
    z_lanes = np.array([lane_coordinate(copy) for copy in Z_ROW_COPIES])
    out["pred_x_projection_cm"] = x_counts.dot(x_lanes) / np.maximum(x_counts.sum(axis=1), EPS)
    out["pred_z_projection_cm"] = z_counts.dot(z_lanes) / np.maximum(z_counts.sum(axis=1), EPS)
    out["dataset"] = dataset
    return out


training_pred = add_centroid_predictions(training, "training")
test_pred = add_centroid_predictions(test, "test")
predictions = pd.concat([training_pred, test_pred], ignore_index=True)

## Summary Metrics

In [ ]:
def summarize(df: pd.DataFrame, method: str, pred_x: str, pred_z: str) -> dict:
    err_x = df[pred_x] - df["true_x_cm"]
    err_z = df[pred_z] - df["true_z_cm"]
    err_r = np.hypot(err_x, err_z)
    return {
        "dataset": df["dataset"].iloc[0],
        "method": method,
        "n_events": len(df),
        "mean_err_x_cm": err_x.mean(),
        "sigma_err_x_cm": err_x.std(ddof=1),
        "mean_err_z_cm": err_z.mean(),
        "sigma_err_z_cm": err_z.std(ddof=1),
        "median_err_r_cm": err_r.median(),
        "p68_err_r_cm": err_r.quantile(0.68),
        "p95_err_r_cm": err_r.quantile(0.95),
    }


summary = pd.DataFrame([
    summarize(training_pred, "all_32_2d_centroid", "pred_x_all_centroid_cm", "pred_z_all_centroid_cm"),
    summarize(training_pred, "projection_centroid", "pred_x_projection_cm", "pred_z_projection_cm"),
    summarize(test_pred, "all_32_2d_centroid", "pred_x_all_centroid_cm", "pred_z_all_centroid_cm"),
    summarize(test_pred, "projection_centroid", "pred_x_projection_cm", "pred_z_projection_cm"),
])
summary.to_csv(SUMMARY_CSV, index=False)
predictions.to_csv(PREDICTIONS_CSV, index=False)
summary

## SiPM Geometry

In [ ]:
pos = sipm_positions()
fig, ax = plt.subplots(figsize=(6, 6))
for family, group in pos.groupby("family"):
    ax.scatter(group["x_cm"], group["z_cm"], label=family, s=50)
    for _, row in group.iterrows():
        ax.text(row["x_cm"], row["z_cm"], str(int(row["copy"])), fontsize=8, ha="center", va="bottom")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-55, 55)
ax.set_ylim(-55, 55)
ax.set_xlabel("x [cm]")
ax.set_ylabel("z [cm]")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sipm_geometry.png", dpi=180)

## Plots

In [ ]:
def add_errors(df: pd.DataFrame, pred_x: str, pred_z: str) -> pd.DataFrame:
    out = df.copy()
    out["err_x_cm"] = out[pred_x] - out["true_x_cm"]
    out["err_z_cm"] = out[pred_z] - out["true_z_cm"]
    out["err_r_cm"] = np.hypot(out["err_x_cm"], out["err_z_cm"])
    return out


def plot_method(df: pd.DataFrame, method: str, pred_x: str, pred_z: str):
    view = add_errors(df, pred_x, pred_z)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].scatter(view["true_x_cm"], view[pred_x], s=5, alpha=0.35)
    axes[0].plot([-50, 50], [-50, 50], color="black", lw=1)
    axes[0].set_xlabel("true x [cm]")
    axes[0].set_ylabel("predicted x [cm]")
    axes[0].set_title("x prediction")

    axes[1].scatter(view["true_z_cm"], view[pred_z], s=5, alpha=0.35)
    axes[1].plot([-50, 50], [-50, 50], color="black", lw=1)
    axes[1].set_xlabel("true z [cm]")
    axes[1].set_ylabel("predicted z [cm]")
    axes[1].set_title("z prediction")

    axes[2].hist(view["err_r_cm"], bins=60)
    axes[2].set_xlabel("radial error [cm]")
    axes[2].set_ylabel("events")
    axes[2].set_title("error distribution")
    fig.suptitle(f"{view['dataset'].iloc[0]}: {method}")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{view['dataset'].iloc[0]}_{method}_summary.png", dpi=180)
    return fig, axes


plot_method(test_pred, "projection_centroid", "pred_x_projection_cm", "pred_z_projection_cm");
plot_method(test_pred, "all_32_2d_centroid", "pred_x_all_centroid_cm", "pred_z_all_centroid_cm");

In [ ]:
def plot_error_map(df: pd.DataFrame, method: str, pred_x: str, pred_z: str):
    view = add_errors(df, pred_x, pred_z)
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(view["true_x_cm"], view["true_z_cm"], c=view["err_r_cm"], s=16, cmap="viridis")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("true x [cm]")
    ax.set_ylabel("true z [cm]")
    ax.set_title(f"{view['dataset'].iloc[0]} radial error: {method}")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("radial error [cm]")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{view['dataset'].iloc[0]}_{method}_error_map.png", dpi=180)
    return fig, ax


plot_error_map(test_pred, "projection_centroid", "pred_x_projection_cm", "pred_z_projection_cm");
plot_error_map(test_pred, "all_32_2d_centroid", "pred_x_all_centroid_cm", "pred_z_all_centroid_cm");